# Nghiên cứu thực nghiệm retrieval cho trợ lý gọi món CMC Restaurant

**Mục tiêu.** So sánh BM25, multilingual E5 dense retrieval và hybrid RRF trên cùng corpus, cùng dev split và cùng implementation metric để chọn ứng viên tích hợp production.

**Phạm vi dữ liệu.** Corpus có 91 món chính thức, bao gồm đồ uống, cộng 35 knowledge document. Notebook chỉ đọc kết quả dev đã sinh từ worktree sạch; frozen test 235 case chưa được mở.

> Đây là model-selection study, không phải bằng chứng test cuối hoặc đánh giá end-to-end chất lượng câu trả lời LLM.

In [ ]:
from __future__ import annotations

import json
import sys
from pathlib import Path

cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parents[1] if cwd.name == 'notebooks' else cwd
AI_ROOT = PROJECT_ROOT / 'ai'
if str(AI_ROOT) not in sys.path:
    sys.path.insert(0, str(AI_ROOT))

ARTIFACT_PATH = AI_ROOT / 'evaluation' / 'results' / 'dev_retrieval_summary.v1.json'
artifact = json.loads(ARTIFACT_PATH.read_text(encoding='utf-8'))
assert artifact['corpus']['menu_items'] == 91
assert artifact['corpus']['includes_drinks'] is True
assert artifact['frozen_test_opened'] is False
assert artifact['provenance']['git_dirty'] is False
print(f"Loaded {ARTIFACT_PATH.relative_to(PROJECT_ROOT)}")
print(f"Git SHA: {artifact['provenance']['git_sha']}")

## 1. Câu hỏi nghiên cứu và giả thuyết

- **RQ1:** Dense retrieval có cải thiện ranking quality so với lexical BM25 trên truy vấn tiếng Việt, không dấu và paraphrase không?
- **RQ2:** Hybrid RRF có bổ sung tín hiệu lexical + semantic để tăng Hit/MRR/nDCG mà vẫn giữ latency dưới ngưỡng tương tác không?
- **RQ3:** Cải thiện quan sát được có bền vững sau confidence interval và Holm-Bonferroni không?

Giả thuyết trước thí nghiệm: `hybrid > dense > BM25` về point estimate quality; BM25 nhanh nhất; hybrid chỉ được chọn nếu không có forbidden hit, p95 retrieval < 25 ms và cải thiện quality so với BM25 còn ý nghĩa sau hiệu chỉnh.

## 2. Thiết kế thí nghiệm và kiểm soát rò rỉ

- Ba phương pháp dùng cùng 126 document, cùng 110 dev query có expected selector và cùng cutoff 1/3/5/10.
- Dev/test được tách vật lý. Dev-run không parse test labels; hai test artifact được khóa SHA-256.
- Dense dùng `intfloat/multilingual-e5-small` revision cố định, prefix `query:`/`passage:`, vector chuẩn hóa 384 chiều.
- Latency: warm-up tối đa 5 query/target, 7 lần đo/query, lấy median; case/method order được shuffle bằng seed cố định.
- Paired bootstrap 10.000 vòng cho MRR/nDCG; McNemar exact cho hit; Wilcoxon signed-rank cho rank/latency; Holm-Bonferroni trong từng test family qua ba cặp phương pháp.
- Công thức metric/statistics nằm trong `ai/evaluation`; notebook không định nghĩa lại.

In [ ]:
print('Dataset/corpus provenance')
print('-' * 72)
print(f"Dev cases: {artifact['dataset_cases']['dev']} (evaluated: {artifact['evaluated_cases']})")
print(f"Frozen test cases: {artifact['dataset_cases']['frozen_test']} (opened: {artifact['frozen_test_opened']})")
print(f"Documents: {artifact['corpus']['documents']} = {artifact['corpus']['menu_items']} menu + {artifact['corpus']['knowledge_documents']} knowledge")
print(f"Corpus SHA-256: {artifact['corpus']['corpus_sha256']}")
print(f"Dev JSONL SHA-256: {artifact['dataset']['materialized_cases_sha256']}")
print(f"Frozen test JSONL SHA-256: {artifact['dataset']['frozen_test_cases_sha256']}")

## 3. Ma trận phương pháp

| Phương pháp | Tín hiệu | Tham số khóa | Kỳ vọng |
| --- | --- | --- | --- |
| BM25 | exact token/term frequency | k1, b, title/tag boost | nhanh, mạnh với tên món chính xác |
| Dense E5 | cosine semantic | model revision, 384d, normalized | mạnh với paraphrase/không dấu |
| Hybrid RRF | BM25 rank + E5 rank | RRF k=60, weight 1:1 | cân bằng lexical và semantic |

In [ ]:
columns = ('method', 'Hit@1', 'Hit@5', 'Hit@10', 'MRR@5', 'nDCG@5', 'forbidden@10', 'p50 ms', 'p95 ms')
rows = []
for method, values in artifact['methods'].items():
    rows.append((
        method, values['hit_at_1'], values['hit_at_5'], values['hit_at_10'],
        values['mrr_at_5'], values['ndcg_at_5'], values['forbidden_at_10'],
        values['p50_ms'], values['p95_ms'],
    ))
widths = (13, 8, 8, 8, 8, 9, 14, 9, 9)
print(' '.join(str(value).ljust(width) for value, width in zip(columns, widths, strict=True)))
print('-' * sum(widths))
for row in rows:
    formatted = (row[0], *(f'{value:.4f}' for value in row[1:7]), *(f'{value:.2f}' for value in row[7:]))
    print(' '.join(str(value).ljust(width) for value, width in zip(formatted, widths, strict=True)))

In [ ]:
bm25 = artifact['methods']['bm25']
dense = artifact['methods']['dense_e5']
hybrid = artifact['methods']['hybrid_rrf']
for label, method in [('Dense E5', dense), ('Hybrid RRF', hybrid)]:
    mrr_gain = 100 * (method['mrr_at_5'] / bm25['mrr_at_5'] - 1)
    ndcg_gain = 100 * (method['ndcg_at_5'] / bm25['ndcg_at_5'] - 1)
    print(f'{label}: MRR@5 {mrr_gain:+.1f}%, nDCG@5 {ndcg_gain:+.1f}% vs BM25')

## 4. Kiểm định thống kê

Delta luôn được định nghĩa `method_a - method_b`. CI là 95%; p-value bên dưới đã qua Holm-Bonferroni. Point estimate không được gọi là cải thiện có ý nghĩa nếu CI/p-value không ủng hộ kết luận.

In [ ]:
for comparison, values in artifact['pairwise_statistics'].items():
    if comparison == 'correction':
        continue
    print(f'\n{comparison}')
    print(f"  MRR delta={values['mrr_delta']:+.4f}, CI={values['mrr_ci95']}, Holm p={values['mrr_holm_p']:.4g}")
    print(f"  nDCG delta={values['ndcg_delta']:+.4f}, CI={values['ndcg_ci95']}, Holm p={values['ndcg_holm_p']:.4g}")
    print(f"  Hit delta={values['hit_delta']:+.4f}, CI={values['hit_ci95']}, Holm p={values['hit_holm_p']:.4g}")
    print(f"  Rank-biserial={values['rank_biserial']:+.4f}, Holm p={values['rank_holm_p']:.4g}")
    print(f"  Latency median delta={values['latency_median_delta_ms']:+.2f} ms, CI={values['latency_ci95_ms']}, Holm p={values['latency_holm_p']:.4g}")

## 5. Decision rule và kết luận dev

Hybrid chỉ được chọn nếu: (1) đứng đầu MRR/nDCG point estimate; (2) tăng quality có ý nghĩa so với BM25; (3) Hit@10 đạt 100%; (4) forbidden hit bằng 0; (5) p95 retrieval dưới 25 ms; (6) frozen test chưa bị dùng để tuning.

In [ ]:
hybrid_vs_bm25 = artifact['pairwise_statistics']['hybrid_rrf_vs_bm25']
decision_checks = {
    'best_mrr': hybrid['mrr_at_5'] == max(v['mrr_at_5'] for v in artifact['methods'].values()),
    'best_ndcg': hybrid['ndcg_at_5'] == max(v['ndcg_at_5'] for v in artifact['methods'].values()),
    'mrr_significant_vs_bm25': hybrid_vs_bm25['mrr_holm_p'] < 0.05,
    'ndcg_significant_vs_bm25': hybrid_vs_bm25['ndcg_holm_p'] < 0.05,
    'perfect_hit_at_10': hybrid['hit_at_10'] == 1.0,
    'no_forbidden_hits': hybrid['forbidden_at_10'] == 0.0,
    'latency_budget': hybrid['p95_ms'] < 25.0,
    'test_remains_closed': artifact['frozen_test_opened'] is False,
}
for check, passed in decision_checks.items():
    print(f"[{'PASS' if passed else 'FAIL'}] {check}")
assert all(decision_checks.values())
print('\nProvisional selection: hybrid_rrf')

## 6. Giới hạn, error analysis và bước tiếp theo

1. Kết quả chỉ đo retrieval, chưa đo factuality, từ chối món lặp, số lượng thẻ gợi ý hay memory continuity của LLM.
2. Latency là warm single-machine benchmark; production cần đo thêm concurrency, cache miss và cold start.
3. Hybrid cao hơn dense về point estimate nhưng quality delta chưa có ý nghĩa sau Holm; không được tuyên bố hybrid thắng dense một cách tổng quát.
4. Dataset là engineering-reviewed; trước production rộng cần restaurant-domain review các family quan trọng.
5. Bước tiếp theo: tích hợp hybrid sau một interface production, thêm table-session memory/rejection state, chạy behavior evaluation, rồi mới mở frozen test đúng một lần theo decision rule đã khóa.